In [2]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
print("✅ Working directory set to project root:", PROJECT_ROOT)


✅ Working directory set to project root: /home/robert/projects/hackaton_test


In [3]:
import pandas as pd

In [4]:
DATAFOLDER = PROJECT_ROOT / "data"
RAW_FOLDER = DATAFOLDER / "raw"
EXTRACTED_FOLDER = DATAFOLDER / "extracted"
RAW_FOLDER.mkdir(parents=True, exist_ok=True)
EXTRACTED_FOLDER.mkdir(parents=True, exist_ok=True)
print("✅ Data folders set up at:", DATAFOLDER)

✅ Data folders set up at: /home/robert/projects/hackaton_test/data


In [5]:
FILES = {
    "2025_Q1": RAW_FOLDER / "personal_wellbeing_loneliness_q1_2025.xlsx",
    "2025_Q2": RAW_FOLDER / "personal_wellbeing_loneliness_q2_2025.xlsx",
    "2025_Q3": RAW_FOLDER / "personal_wellbeing_loneliness_q3_2025.xlsx",
    "2025_Q4": RAW_FOLDER / "personal_wellbeing_loneliness_q4_2025.xlsx",
}

In [6]:
METRICS = {
    "Table_1": "life_satisfaction",
    "Table_2": "worthwhile",
    "Table_3": "happiness",
    "Table_4": "anxiety",
}

In [7]:
SKIPROWS = 9  # keep consistent with what you used

In [8]:
def extract_region_mean(path: Path, sheet: str, metric_name: str, quarter: str) -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name=sheet, skiprows=SKIPROWS)

    # Filter to Region rows only
    df = df[df["Breakdown category"] == "Region"][["Breakdown", "Estimate (mean score out of 10)"]].copy()

    # Rename + clean
    df.columns = ["region", metric_name]
    df["region"] = df["region"].astype(str).str.strip()
    df[metric_name] = pd.to_numeric(df[metric_name], errors="coerce")
    df["quarter"] = quarter

    # Sanity checks (helpful during hackathon)
    if df["region"].nunique() != 9:
        print(f"⚠️ {quarter} {sheet}: expected 9 regions, got {df['region'].nunique()}")

    return df[["quarter", "region", metric_name]]

In [9]:
def build_quarter_means(path: Path, quarter: str) -> pd.DataFrame:
    quarter_df = None

    for sheet, metric_name in METRICS.items():
        m = extract_region_mean(path, sheet, metric_name, quarter)

        if quarter_df is None:
            quarter_df = m
        else:
            quarter_df = quarter_df.merge(m, on=["quarter", "region"], how="inner")

    # Ensure consistent column order
    return quarter_df[["quarter", "region", "life_satisfaction", "worthwhile", "happiness", "anxiety"]]


In [10]:
all_quarters = [build_quarter_means(path, q) for q, path in FILES.items()]
final_means = pd.concat(all_quarters, ignore_index=True)
final_means["wellbeing_index"] = (
    final_means["life_satisfaction"] +
    final_means["worthwhile"] +
    final_means["happiness"] -
    final_means["anxiety"]
) / 4
final_means["wellbeing_index"] = final_means["wellbeing_index"].round(2)

In [11]:
print(final_means.shape)          # should be (36, 6)
print(final_means.head())

(36, 7)
   quarter                    region  life_satisfaction  worthwhile  \
0  2025_Q1                North East                6.9         7.4   
1  2025_Q1                North West                6.9         7.3   
2  2025_Q1  Yorkshire and The Humber                6.9         7.0   
3  2025_Q1             East Midlands                6.8         7.2   
4  2025_Q1             West Midlands                6.8         7.1   

   happiness  anxiety  wellbeing_index  
0        7.0      3.9             4.35  
1        6.9      3.7             4.35  
2        7.1      3.9             4.28  
3        7.0      4.0             4.25  
4        6.9      3.9             4.22  


In [12]:
# Save one clean file
OUT_PATH = EXTRACTED_FOLDER / "regional_means_quarterly.csv"
final_means.to_csv(OUT_PATH, index=False)
print(f"Saved: {OUT_PATH}")

Saved: /home/robert/projects/hackaton_test/data/extracted/regional_means_quarterly.csv


In [13]:
final_means.shape        

(36, 7)

In [14]:
          # should be (36, 6)
final_means["quarter"].nunique()   

4

In [15]:
# should be 4
final_means.groupby("quarter")["region"].nunique()  # should be 9 for each


quarter
2025_Q1    9
2025_Q2    9
2025_Q3    9
2025_Q4    9
Name: region, dtype: int64

In [16]:
final_means[["life_satisfaction","worthwhile","happiness","anxiety"]].describe()

,life_satisfaction,worthwhile,happiness,anxiety
count,36.000000,36.000000,36.000000,36.000000
mean,6.838889,7.205556,6.975000,3.958333
std,0.120185,0.135107,0.136015,0.169664
min,6.600000,6.900000,6.700000,3.600000
25%,6.700000,7.175000,6.900000,3.900000
50%,6.800000,7.200000,7.000000,3.950000
75%,6.900000,7.300000,7.100000,4.025000
max,7.100000,7.400000,7.200000,4.500000


In [17]:
# should be 4
final_means[["life_satisfaction","worthwhile","happiness","anxiety","wellbeing_index"]].corr()

,life_satisfaction,worthwhile,happiness,anxiety,wellbeing_index
life_satisfaction,1.000000,0.654946,0.742822,-0.450712,0.869567
worthwhile,0.654946,1.000000,0.722970,-0.301218,0.816012
happiness,0.742822,0.722970,1.000000,-0.281669,0.836656
anxiety,-0.450712,-0.301218,-0.281669,1.000000,-0.680319
wellbeing_index,0.869567,0.816012,0.836656,-0.680319,1.000000
